Si distinguono due modi di descrivere un sistema distribuito:
- **Architettura Software**: descrive l'organizzazione logica, sarebbe a dire che componenti costituiscono l'applicazione, quali responsabilità hanno e come interagiscono tra loro.
- **Architettura di Sistema**: descrive come l'architettura software è concretamente realizzata sulle risorse disponibili, incudendo il **deployment** (ossia il collocamento e l'avvio dei componenti sull'infrastruttura)

Ad es. un'applicazione può avere tre microservizi: utenti, catalogo e ordini. Mentre l'architettura software descrive le loro funzioni e interazioni, quella di sistema stabilisce su quali macchine vengono eseguiti, quante istanze vengono avviate e come sono distribuite.  
In generale, in caso di sovraccarico/guasto, questa collocazione può essere modificata per garantire la continuità del servizio.

## Architettura Software
Uno **stile architetturale** è un insieme di scelte progettuali che definisce come organizzare i componenti e le loro interazioni. Due elementi fondamentali:
- **Componente** == unità modulare con un'interfaccia ben definita (es. servizio che gestisce gli ordini)
- **Connettore** == meccanismo che permette ai componenti di comunicare tra loro (es. chiamata HTTP, messaggi su coda, ecc.)

Un componente può esser sostituito con un'altra implementazione se questa mantiene un'interfaccia e un comportamento compatibili con ciò che gli altri componenti si aspettano.  
Lo stile architetturale stabilisce inoltre quali componenti possono interagire, quali dati si scambiano e come sono configurati nel loro insieme.

Tre famiglie principali di stili: **a livelli**, **orientata ai servizi** e **publish-subscribe**.

### Stile a livelli
Nello stile **layered**, i componenti sono organizzati a livelli, ognuno con responsabilità specifiche.  
Nel modello tradizionale un livello superiore usa i servizi di un livello inferiore:
- la richiesta scende a livelli sottostanti
- il risultato sale al chiamante

Sia $N$ il livello superiore e $1$ quello più basso. Allora un componente nel livello $i$ invoca un componente nel livello $j$ se e solo se $j < i$.

Il vantaggio più grande è la **separazione di responsabilità**: ogni livello si concentra su un compito ed espone agli altri ciò che serve attraverso un'interfaccia.

Esistono anche varianti dove le chiamate possono essere fatte saltando alcuni livelli, possono esserci chiamate senza risposta immediata (one-way call) e comunicazioni verso l'alto (upcall)

<img src="imgs/layered.png" alt="Layered Architecture" width="500"/>

Per quel che riguarda un'applicazione, la suddivisione tradizionale presenta quattro livelli:
1. **Presentation Layer**: gestisce l'interfaccia utente e la presentazione dei dati.
2. **Business Logic Layer**: contiene la logica di business e le regole dell'applicazione.
3. **Persistence Layer**: gestisce l'accesso ai dati conservati
4. **Database Layer**: rappresenta il livello più basso, dove i dati sono effettivamente memorizzati. Qui quindi si memorizzano e interrogano i dati.

es. in un negozio online il presentation layer mostra il carrello, la logica applicativa calcola il totale, il livello di persistenza gestisce le operazioni di salvataggio e il database conserva l'ordine effettuato.

Persistenza e Database sono distinti perché il primo contiene il codice che accede ai dati, mentre il secondo è proprio il sistema che li conserva.  
NB questi sono livelli logici, non devono essere eseguiti in macchine diverse! Inoltre in alcune implementazioni alcuni livelli possono essere accorpati.

### Stile orientato ai servizi
L'applicazione è vista come una raccolta di entità separate, **ciascuna delle quali incapsula un servizio.**  
Ogni entità realizza una funzione ed espone un'interfaccia attraverso cui gli altri componenti possono utilizzarla, nascondendo l'implementazione interna.

Tre declinazioni principali: a Oggetti, a Microservizi e RESTful. Queste descrizioni però possono anche combinarsi, ad esempio un microservizio può esporre un'interfaccia REST.

#### Stile basato su oggetti
Ogni componente è un oggetto che contiene dati e offre metodi per operarvi.  
Ciò permette di applicare i principi di:
- **incapsulamento**: i dati sono nascosti e accessibili solo tramite metodi pubblici
- **information hiding**: i dettagli interni sono nascosti
- **riuso**: gli oggetti possono essere riutilizzati in contesti diversi

In questo senso si cità il riutilizzo di sistemi **legacy**: un componente può essere integrato in un nuovo sistema senza modificarne l'implementazione, purché l'interfaccia sia compatibile.

La comunicazione tra oggetti può avvenire tramite **RPC Remote Procedure Call**: un componente richiede l'esecuzione di una procedura su un altro processo, eventualmente su un'altra macchina. Il meccanismo nasconde parte dei dettagli della comunicazione, ma la chiamata resta soggetta a latenza e possibili errori di rete.

#### Stile a microservizi
L'applicazione viene scomposta in servizi relativamente piccoli, autonomi e dedicati a responsabilità specifiche. Questi comunicano tramite API ben definite.  

Con questo stile si vuole che tali servizi siano **loosely coupled**: si cerca di limitare le dipendenze tra i servizi. Proprietà fondamentali:
- possibilità di sviluppare e distribuire i servizi separatamente
- possibilità di aggiornarli senza ridistribuire tutta l'applicazione
- possiiblità di scalare ciascun servizio in base al carico

es. se aumentano le richieste al servizio ordini, si possono aggiungere istanze solo quello senza toccare gli altri. 

L'indipendenza in realtà resta limitata alle interazioni: un servizio dovrà quasi sempre comunque dipendere dalle risposte o dai dati di un altro.  
Inoltre attenzione al fatto che dividere troppo finemente l'applicazione può aumentare la complessità. 

es. DeathStarBench: si mostra un'applicazione con funzionalità relative a utenti, film, valutazioni e recensioni. Gli aspetti più importanti in un'architettura complessa del genere sono
- API gateway: il punto di ingresso che riceve le richieste e le indirizza ai microservizi appropriati
- Gestione separata dei dati e dei servizi: ogni microservizio tende a gestire i dati di propria competenza e può scegliere una tecnologia adatta alle sue esigenze (es. database relazionale, NoSQL, ecc --> **database per service**) (tipo per un microservizio serve database persistente che quindi conservi dati anche dopo la chiusura, e magari affiancata una cache veloce, per un altro un database in-memory tipo Redis)

#### Stile RESTful
**REST (Representational State Transfer)** è uno stile architetturale per sistemi distribuiti dove **i componenti possono operare attraverso un'interfaccia uniforme e stateless**. Negli esempi che vedremo, viene utilizzato **HTTP**.

Una risorsa può essere un utente, una recensione, un documento o un oggetto conservato nel cloud. Le risorse sono identificate tramite **URI Uniform Resource Identifier**, mentre un **URL Uniform Resource Locator** è un tipo di URI che indica anche come localizzare e accedere alla risorsa.

L’interfaccia uniforme permette di interagire con risorse diverse attraverso operazioni e convenzioni comuni. I messaggi devono fornire informazioni sufficienti per essere interpretati, tramite contenuto e metadati.

Con **stateless** si intende che ogni richiesta deve contenere le informazioni necessarie a comprenderla ed eseguirla, senza che il server debba ricordare il contesto delle precedenti richieste del client.  
**Attenzione che stateless non significa che il server non conservi dati**: un server può conservare utenti, ordini e recensioni in un database. Ciò che non deve essere indispensabile è mantenere conversazioni precedenti per interpretare le richieste successive.

Ripassiamo le operazioni REST con metodi HTTP:
- **GET**: recupera una risorsa senza modificarla
- **PUT**: crea o sostituisci lo stato della risorsa all'URI specificato
- **POST**: crea una nuova risorsa o invia dati al server per elaborazione
- **DELETE**: rimuove la risorsa all'URI specificato

Differenza tra PUT e POST: PUT specifica lo stato desiderato della risorsa (es. aggiorna la recensione del film mettendo voto 7), mentre POST invia dati al server per creare una nuova risorsa (es. invia una recensione con voto 7 e commento "bello"). PUT è idempotente: se invio più volte la stessa richiesta, il risultato sarà lo stesso. POST invece può generare risorse diverse a seconda del numero di richieste.

es. Amazon S3: è un servizio di storage a oggetti. Gli oggetti sono conservati in bucket e identificati da una chiave. 
```plaintext
https://example-bucket.s3.us-west-2.amazonaws.com/photos/puppy.jpg
```
in questo esempio si distingue il bucket `example-bucket`, la regione `us-west-2` e la chiave dell'oggetto `photos/puppy.jpg`.  
Quindi `photos/` è parte della chiave dell'oggetto, anche se l'interfaccia può mostrarla come una cartella.

Le operazioni sono poi tradotte in richieste HTTP: GET sul bucket per elencarne gli oggetti, GET sull'oggetto per recuperarlo, PUT per caricare un oggetto.  
La richiesta inoltre contiene informazioni per l'autenticazione (ricorda autenticazione != autorizzazione: la prima verifica chi presenta la richiesta, la seconda se può compiere l'operazione).

### Decoupling
Quando un componente chiama direttamente un altro, possono nascere dipendenze; deve infatti conoscerne l'identità/indirizzo, può richiedere che sia disponibile nello stesso momento, può restare bloccato aspettando la risposta...

Queste dipendenze costituiscono forme di accoppiamento, **coupling** tra componenti. Per ridurle si introduce un **intermediario**, per esempio una coda o un sistema di messaggistica. 

In questo modo si separano maggiormente sia la **computazione** (lavoro dei componenti) che il loro **coordinamento** (come interagiscono tra loro).  
Ciò può rendere più semplice sostituire componenti, aggiungere istanze e gestire indisponibilità temporanee (better scalability and elasticity). Tuttavia è necessario che l'intermediario offra tutte le proprietà necessarie.

Esistono tre **proprietà del decoupling**:
1. **Space Decoupling**: i componenti non devono conoscere l'identità reciproca (es. un produttore scrive su una coda senza conoscere quale consumatore leggerà)
2. **Time Decoupling**: i componenti non devono essere attivi nello stesso momento (è necessario che il messaggio venga conservato finché il destinatario torna disponibile)
3. **Synchronization Decoupling**: i componenti non devono attendersi reciprocamente, restando bloccati (es. il mittente invia una richiesta e prosegue con un altro lavoro senza attendere la risposta)

Nel Decoupling spaziale, i componenti conoscono il punto di comunicazione comune, ma non chi c'è dall'altra parte.  
In quello temporale, l'intermediario deve poter conservare il messaggio. In questo modo il mittente può inviarlo anche quando il destinatario è inattivo.  
Nel Decoupling di sincronizzazione, chi avvia l’interazione può continuare l’esecuzione senza attendere immediatamente il completamento del lavoro dell’altro componente.

Queste tre proprietà sono tutte distinte: ad es. una chiamata asincrona può evitare il blocco del mittente, ma richiede comunque che il destinatario sia disponibile in quel momento, per eliminare questa dipendenza serve un meccanismo che conservi la comunicazione etc..

Altro concetto importante da ricordare è la differenza tra interazione **sincrona** e **asincrona** tra componenti.

Nell'**interazione sincrona**, il chiamante invoca una procedura remota e rimane bloccato finché il chiamato non resituisce il risultato (es. un componente chiede di calcolare il totale dell'ordine e attende la risposta).  
Qui il tempo di attesa comprende trasferimento richiesta, elaborazione e trasferimento risposta.

Nell'**interazione asincrona** invece è introdotto un intermediario: una coda oppure un **MOM** (Message-Oriented Middleware, un sistema che gestisce la comunicazione tra componenti tramite messaggi).  
In questo caso il producer invia msg all'intermediario, può quindi continuare a svolgere il proprio lavoro. Il consumer riceve il messaggio dall'intermediario e lo elabora, e un'eventuale risposta viene comunicata al producer tramite l'intermediario (si evita polling per cui il producer non deve continuamente chiedere se il consumer ha finito).  
Inoltre si distinguono due modalità di consegna: **Pull** se è il destinatario a chiedere/prelevare il messaggio all'intermediario, o **Push** se è l'intermediario a inviare il messaggio/notifica al destinatario.


<img src="imgs/syncasync.png" width="400"/>


#### Vantaggi e costi del decoupling
Il decoupling rende il sistema più flessibile dal momento che riduce alcune dipendenze tra componenti. Ad esempio:
- Spaziale --> il vantaggio è che un componente può essere sostituito, aggiornato, replicato o spostato senza che i mittenti debbano conoscerne la nuova collocazione
- Temporale --> il mittente e il destinatario possono essere disponibili in momenti diversi
- di Sincronizzazione --> un componente può proseguire senza aspettare il lavoro di un altro

Il decoupling temporale è utile con componenti **volatili** (i.e. entrano ed escono dal sistema o non sono sempre attivi, come applicazioni serverless). 

Tuttavia tutta questa flessibilità ha un costo: serve un intermediaro e quindi un'infrastruttura che **introduce elaborazione, trasferimenti, memorizzazione e gestione aggiuntivi --> può aumentare latenza e costo complessivo**.

Il compromesso è tra **benefici di disaccoppiamento** e **overhead di intemediazione** (vedremo Kafka anche per capire come contenere questo overhead).

Gli stili basati su comunicazione indiretta sono principalmente 3: **Event-driven**, **Data-oriented**, **Publish-subscribe**.

#### Stile Event-driven
I componenti comunicano attraverso **eventi**, dove un evento rappresenta un cambiamento di stato significativo per l'applicazione (es. variazione temperatura, apertura di una porta, disponibilità di un'immagine da elaborare...).

Per questo stile le componenti comunicano attraverso un **event bus**: alcune componenti **pubblicano gli eventi**, altre si **iscrivono per ricevere quelli di interesse**, e il bus si occupa di inoltrare le notifiche.  
Es. ridimensionamento immagini: un componente pubblica l'evento "immagine disponibile", il componente che ridimensiona le immagini riceve l'evento, recupera l'immagine e la elabora (quindi l'evento può ottenere il riferimento all'immagine, non per forza il file)

La comunicazione è:
- **anonima tra produttore e destinatari** --> non devono conoscersi direttamente --> **decoupling spaziale**.  
- **asincrona** --> i componenti possono proseguire lavoro mentre attendono notifiche --> **decoupling di sincronizzazione**.
- **multicast** --> un evento può essere ricevuto da più destinatari (multicast indica la distribuzione logica a più componenti, non implica necessariamente l'uso di multicast di rete)

E il **decoupling temporale**? Dipende dall'implementazione dell'event bus: *se il bus può conservare gli eventi finché i destinatari non sono pronti, allora sì*. Altrimenti no.  
Nel modello di base non è previsto che il bus conservi gli eventi, quindi **Event-Driven non offre generalmente decoupling temporale**.

#### Stile Data-oriented
In questo stile i componenti comunicano attraverso uno **spazio dati condiviso**. Un componente deposita il dato mentre l'altro lo recupera: non occorre quindi comunicazione diretta tra i due. 

Operazioni principali `write`, `read` e `take` (che legge e rimuove il dato dallo spazio condiviso). Anche varianti come `readIfExists` e `takeIfExists` (verificano la presenza del dato senza doverlo attendere indefinitamente).

Dato che `take` rimuove il dato, se più componenti lo leggono, solo uno lo otterrà --> in generale **è necessario gestire la concorrenza**. 

Si distinguono inoltre due tipi di spazio:
- **Spazio passivo**: risponde solo alle operazioni dei componenti
- **Spazio attivo**: può inviare notifiche o effettuare push quando diventano disponibili dati di interesse

Con spazio passivo un componente può dover attendere una lettura bloccante o **fare polling**, le notifiche evitano queste attese/controlli ripetuti.

Quindi Data-oriented offre **decoupling spaziale** (le componenti conoscono lo spazio, ma non si conoscono tra loro) e **temporale** (lo spazio conserva i dati, quindi i componenti non devono essere attivi contemporaneamente).  
Al contrario il **decoupling di sincronizzazione** dipende dalle API e dal meccanismo di notifica: uno spazio passivo può comportare che un mittente resti bloccato in attesa di un dato da parte del destinatario. Invece uno spazio attivo con giusta implementazione delle notifiche può garantire questo decoupling.

#### Stile Publish-subscribe
Nel modello **pub/sub** esistono due ruoli:
- **Publisher**: colui che produce e pubblica eventi
- **Subscriber**: colui che si iscrive per ricevere eventi di interesse

Uno stesso componente può svolgere entrambi i ruoli. Esiste un **Middleware pub/sub** che mantiene le iscrizioni e gestisce la distribuzione --> il publisher non deve individuare tutti i destinatari: pubblica l'evento e affida al middleware la consegna. 

<img src="imgs/pubsub.png" width="400"/>

Con questo stile si ottengono **tutti i decoupling**: spaziale perché sub e pub non devono conoscersi, temporale perché il middleware può **conservare** gli eventi finché i destinatari non sono pronti, e di sincronizzazione perché il mittente può proseguire senza attendere la risposta del destinatario.

Perché diverso dal Data-Oriented con spazio attivo? Logicamente sono uguali, la differenza sta nell'implementazione (in data oriented enfasi sul depositare, leggere e prelevare dati, in pub/sub enfasi sul pubblicare eventi e sottoscriversi alla loro ricezione).

##### Modelli di sottoscrizione
Esistono due modelli principali per far sì che il Middleware decida quali eventi consegnare a ciascun subscriber: **Topic Based** e **Content Based**.

Nel **Topic Based**, gli eventi appartengono a degli argomenti (topic) e i subscriber si iscrivono a uno o più topic. Il middleware inoltra gli eventi ai subscriber iscritti al topic corrispondente.  
Questo modello è semplice e veloce, poiché è sufficiente confrontare il topic dell'evento con quelli a cui il subscriber è iscritto per capire se inoltrarglielo. Tuttavia offre anche una selezione relativamente poco espressiva: un subscriber riceverà tutti gli eventi di un topic, anche quelli che non lo interessano.

Nel **Content Based** invece il subscriber può specificare **filtri più complessi** basati sul contenuto dell'evento (es. `categoria = "elettronica" AND prezzo < 100`). Il middleware deve verificare per ogni subscriber se l'evento soddisfa il filtro prima di inoltrarlo.  
Questo modello è molto più espressivo **ma rende anche costoso il matching, comportando quindi problemi di scalabilità all'aumentare del numero di subscriber e della complessità dei filtri**.

In pratica quindi il Topic Based è il modello più diffuso in quanto più adatto a sistemi orientati a throughput (i.e. molti eventi al secondo) elevati, come vedremo con Kafka.  
In questo caso quindi **compromesso tra espressività e scalabilità**.


In generale, **non esiste uno stile architetturale migliore per tutte le applicazioni**. La scelta oltre a dipendere dai requisiti specifici dell'applicazione (es. requisiti di performance, di sicurezza, ecc.) dipende anche dai **requisiti non funzionali** come costi, scalabilità ed elasticità, prestazioni, tolleranza ai guasti, manutenibilità, facilità di uso e riuso etc...

Un sistema reale può anche **combinare più stili architetturali assegnandoli alle parti per le quali sono più adatti**. 

Ad esempio per la parte di API esposte al client uno stile possibile è il REST, per le elaborazioni che vengono avviate periodicamente in base alla disponibilità dei dati invece uno stile candidato è Event-Driven o Pub/Sub, per la comunicazione interna potrei realizzare le componenti come microservizi facendoli interagire tramite RPC, per l'analisi e raccolta di log invece uno stile Data-Oriented con spazio attivo può essere più adatto.

## Architettura di Sistema
Finora abbiamo descritto l'organizzazione logica dei componenti, qui si considera invece la loro **istanziazione a runtime sull'infrastruttura** (i.e. **deployment**):
- quali componenti eseguire
- come collegarli concretamente
- su quali nodi collocarli (questa scelta è chiamata **placement**. Un esempio è lo scheduler di Kubernetes)

Le famiglie che introduciamo sono **centralizzate**, **decentralizzate** o **ibride**.

### Architetture Centralizzate, modello client-server
Nel modello client-server i server offrono i servizi, mentre i client ne richiedono l'utilizzo.

L'approccio tipico è il **request-reply**: il client invia una richiesta al server e attende la risposta, quindi spesso comunicazione sincrona e bloccante (evidente dipendenza del client dal server: se il server non risponde, il client resta bloccato).

Attenzione al fatto che ovviamente **centralizzato** non significa necessariamente "eseguito su una sola macchina": un servizio può avere molti server e restare logicamente centralizzato.

#### Tier e Layer

Si distinguono ora:
- **Layer == livello logico** (separazione in layer == separazione logica delle responsabilità)
- **Tier == livello fisico** (separazione in tier == separazione dei componenti nell'esecuzione e nel deployment)

Per quanto riguarda i layer se ne distinguono di tre tipi (sono gli stessi visti all'inizio parlando degli strati applicativi): **Presentation Layer**, **Business Logic Layer** e **Persistence Layer**.  
Ognuno di questi **Layer** può essere distribuito su più **Tier**.

Nella foto vediamo tutte configuarazioni two-tier perché hanno un lato client e uno server. Cambia però la ripartizione dei layer, da quelli con meno responsabilità per il client a quelli in cui invece viene molto "appesantito". In generale ovviamente quindi tre layer logici non implicano tre tier.  

<img src="imgs/two-tier.png" width="400"/>
 
Nella configurazione three-tier tradizionale, presentazione, logica e dati sono separati da tre tier distinti.  
L'architettura però può aggiungere anche altri componenti come cache, load balancer, servizi intermedi di sicurezza (es. WAF Web Application Firewall) etc.. per cui il numero di tier può crescere a $N$. 

Una maggiore separazione permette di specializzare i componenti e **scalare in modo indipendente per ciascun tier**, tuttavia comporta anche **più collegamenti di rete, più componenti da gestire e maggiore complessità**.

#### Distribuzione verticale e orizzontale
Nella **distribuzione verticale**, parti che svolgono funzioni diverse vengono collocate su macchine (nodi) diverse (es. presentation su client, logica su server e dati su database).

Nella **distribuzione orizzontale** uno stesso livello viene distribuito su più nodi (es. più server applicativi gestiscono le richieste, ripartite da un load balancer).

Le due forme possono anche combinarsi: si separano i layer in macchine diverse, ma ognuno o alcuni di essi vengono replicati!  
NB. distribuzione verticale != vertical scaling (quest'ultimo indica l'aumento di risorse su un singolo nodo, es. più CPU o RAM).

<img src="imgs/vertical-horizontal.png" width="300"/>

**Esempio 1: Web application in AWS**

<img src="imgs/aws.png" width="500"/>

La figura realizza un'architettura web a tre tier con distribuzione orizzontale, isolamento di rete e componenti in due Availability Zone AZ.  
La separazione prevede:
- Web Tier: gestisce la parte web e l'inoltro di richieste applicative. Le componenti mostrate sono istanze EC2
- App Tier: esegue la logica dell'applicazione, di nuovo istanze EC2
- Database tier: conserva e interroga i dati relazionali, qui si usa un database Amazon RDS

Mentre EC2 fornisce le VM, RDS è un servizio di database gestito. Il servizio è IaaS (Infrastructure as a Service): la figura combina le VM amministrate dal cliente con un servizio che nasconde la complessità di gestione del database.

Il percorso è il seguente: l'utente raggiunge il load balancer pubblico, che sceglie un'istanza del web tier. L'istanza a sua volta comunica con un altro LB, stavolta **interno all'application tier**, che sceglie un'istanza applicativa. L'istanza allora esegue la logica e accede a RDS quando servono i dati. I risultati tornano infine all'utente.

La prof insiste su tre proprietà:
- **Elasticità**: si ottiene tramite **autoscaling**, il web tier e l'application tier ne hanno due gruppi distinti. Il numero di istanze può quindi aumentare o diminuire in base al carico.
- **Disponibilità**: si ottiene in questo caso tramite più **AZ Availability Zone**. Se un AZ diventa indisponibile, il traffico viene reindirizzato a quello ancora attivo.
- **Sicurezza** attraverso separazione della rete: nell'architettura mostrata si trova infatti una **VPC** (rete virtuale isolata). Mentre il livello web è ancora collocato in subnet pubblico, i livelli applicativi e database sono in subnet private. 

Nella figura si vendono anche il Bastion Host (per accedere alle istanze private dall'amministrazione) e il NAT Gateway (per permettere alle istanze private di accedere a internet senza esporle direttamente).

**Esempio 2: 3-tier serverless architecture**

<img src="imgs/serverless.png" width="450"/>

Anche questa architettura separa presentazione, logica e dati, ma qui i componenti sono **serverless**.

Serverless significa che **non solo il cliente non deve predisporre e amministrare direttamente i server fisici che eseguono il codice, ma nemmeno gli ambienti di esecuzione**: se ne occupa del tutto il provider cloud (es. Con una VM tipo EC2 AWS gestisce solo l'hardware fisico, mentre il cliente si occupa di configurare le VM etc.. mentre con serverless tipo Lambda AWS gestisce anche gli ambienti di esecuzione)

Possiamo dire quindi che con serverless ci si avvicina a un modello **PaaS Platform as a Service** e nel caso specifico di Lambda a un modello **FaaS Function as a Service**, per cui l'astrazione è portata a una singola funzione invocata.

Qui il Presentation Layer è realizzato con i servizi S3, CloudFront e Cognito. Per la logica applicativa API Gateway e Lambds mentre per i dati un database noSQL DynamoDB.

Cognito gestisce l'autenticazione degli utenti, S3 conserva gli asset statici (HTML, CSS, JS) e CloudFront (che è una CDN i.e. rete di distribuzione di contenuti) li distribuisce geograficamente per ridurre la latenza. S3 si occupa di conservare i contenuti del sito.

Per la logica applicativa il frontend del cliente chiama anzitutto l'API Gateway, che inoltra la richiesta alla funzione Lambda associata. A quel punto la funzione esegue il codice backend. Le funzioni Lambda possono leggere e scrivere dati su DynamoDB, che è un database NoSQL gestito.

**Perché non compaiono i gruppi di autoscaling?**  
Non è che non esistono, ma non sono espliciti perché gestiti dal servizio: si occupa Lambda di gestire l'allocazione degli ambienti di esecuzione in risposta alle invocazioni.

Si introduce in questo senso il concetto di **cold start**: quando una funzione non è stata invocata da un po', l'ambiente di esecuzione viene rimosso. Al successivo invocazione, il servizio deve allocare un nuovo ambiente e caricare il codice della funzione, quindi la prima invocazione può richiedere più tempo.

### Architetture Decentralizzate
Nei sistemi P2P le funzionalità del sistema vengono realizzate usando **risorse distribuite e condivise dai nodi partecipanti**, chiamati **peer**. 

Le risorse condivise possono essere di vario tipo: capacità di calcolo, memoria, banda di rete, file... il punto è che un peer può sia utilizzare risorse degli altri che metterne a disposizione.

Caratteristiche fondamentali:
- **Simmetria dei ruoli**: ogni peer può agire sia come client che come server. La simmetria riguarda soprattutto funzioni e responsabilità, non significa che tutti i peer abbiano stessa potenza e comportamento. Si distinguono infatti ad es. i **superpeer**, nodi con compiti aggiuntivi rispetto agli altri
- **Decentralizzazione del controllo**: mentre nel modello centralizzato c'è un solo nodo che governa le operazioni, in P2P il controllo è realizzato con la collaborazione tra peer. 
- **Ampia distribuzione**: i peer possono essere molti e geograficamente dispersi (tipicamente ai margini della rete, poiché sulle macchine dei partecipanti)
- **Dinamismo**: i peer possono entrare e uscire dal sistema in qualsiasi momento.

Si introducono i termini **Membership == insieme dei nodi che fanno parte del sistema in un dato momento** e **Churn == il continuo ingresso e uscita dei nodi, che modifica la membership**. 

Inoltre esiste **ridondanza delle informazioni**: conservando più copie di dati su peer diversi migliora disponibilità e tolleranza ai guasti.

Architetture P2P in moltissimi ambiti:

<img src="imgs/p2p.png" width="400"/>

Nell'esempio di Folding@home, gli utenti mettono a disposizione capacità di calcolo per eseguire parti di simulazioni scientifiche. Ma non è pericoloso eseguire codice di qualcun altro? In questo senso il **modello di fiducia dipende dal sistema!**

#### P2P challenges e compiti di un peer
Le principali sfide dei sistemi P2P sono:
- **Eterogeneità delle risorse**: i peer possono differire di hardware, software, memoria, potenza di calcolo e connessione. Un sistema non può quindi fare assunzioni sulle capacità dei peer, ma deve adattarsi a ciò che offre ciascuno.
- **Scalabilità**: le operazioni devono restare efficienti anche quando aumentano i peer. In questo senso ci concentreremo sul **resource lookup**, i.e. come localizzare una risorsa senza interrogare tutti i nodi.
- **Località e interoperabilità**: una risorsa disponibile non è necessariamente vicina al richiedente. Inoltre due nodi possono essere vicini nella struttura logica del sistema ma molto lontani nella rete sottostante. 
- **Tolleranza ai guasti e al churn**: i peer possono entrare e uscire in qualsiasi momento. In particolare se un peer sta eseguendo un'operazione e si disconnette, il sistema deve essere in grado di recuperare e continuare l'operazione senza perdere dati o corromperli.
- **Prestazioni e auto-organizzazione**: in assenza di un controllore centrale, i peer devono organizzarsi per gestire le connessioni, distribuire il carico e instradare le richieste. 
- **Free-riding**: un free rider usa le risorse del sistema senza contribuire in misura adeguata (es. scarica file senza condividerli). Occorrono meccanismi di incentivo e fairness.
- **Anonimato**: La decentralizzazione non produce automaticamente anonimato: sono necessari meccanismi specifici.  
  Un esempio è Tor che sfrutta **onion routing**: il messaggio attraversa più intermediari ed è protetto da strati di cifratura. Ogni intermediario rimuove il proprio strato e ottiene solo le informazioni necessarie per inoltrare il messaggio al successivo nodo. In questo modo il mittente e il destinatario restano anonimi.
- **Fiducia e reputazione**: i nodi non sempre si conoscono --> un nodo potrebbe fornire informazioni false o risultati errati. Servono quindi meccanismi per valutare l'affidabilità dei peer, a seconda dell'applicazione (es. in bitcoin la "fiducia" è basata sulla PoW Proof of Work)
- **Attacchi Sybil**: nel mondo P2P un nodo può creare più identità false (es. esegue tante VM sulla stessa macchina), ciò potrebbe permettergli di avere un'influenza sproporzionata sul sistema (es. controllare la maggioranza dei nodi in un sistema di voto). 
- **DDoS e altri comportamenti malevoli**: per difendersi serve verifica, ridondanza e **rate limiting** (limitare la frequenza delle richieste)

Consideremo come caso di studio un sistema P2P per la condivisione di file: molti dei meccanismi ideati per risolvere i problemi di questa architettura infatti vedremo che sono stati poi riutilizzati in altri contesti.

Tre operazioni fondamentali:
1. **Bootstrapping**: un nuovo nodo deve scoprire almeno alcuni peer già presenti per iniziare a partecipare. Diverse possibilità, ad esempio indirizzi conosciuti attraverso una configurazione statica, una lista di peer conservata da sessioni precedenti, nodi noti che forniscono info sugli altri partecipanti...
2. **Resource Lookup**: un peer deve essere in grado di localizzare il file desiderato. Questo è il problema principale che approfondiremo
3. **Resource Retrieval**: una volta localizzato il file, il peer deve essere in grado di recuperarlo. Il trasferimento può avvenire da un singolo nodo oppure da più nodi, recuperndo parti diverse del file.

#### Le reti P2P come **overlay network**
Con **overlay network** si intende **una rete logica costruita sopra una rete di comunicazione sottostante**.  
Nella figura in basso le macchine fisiche e la rete di collegamento, al centro la comunicazione TCP/IP e in alto i peer e i collegamenti logici dell'overlay. 

I collegamenti dell'overlay indicano quali peer comunicano direttamente secondo l'organizzazione del sistema P2P. **Un collegamento logico non equivale a un cavo diretto tra le due macchine**: un messaggio può attraversare diversi router e collegamenti della rete sottostante.  
Per questo motivo **due peer adiacenti nell'overlay possono potenzialmente anche trovarsi in continenti diversi**.

<img src="imgs/overlay.png" width="400"/>

L'**overlay routing** serve quindi a trovare un percorso che consenta di **localizzare una risorsa**.  
La differenza rispetto al routing IP riguarda l'obiettivo:
- nel routing IP si instradano pacchetti verso un indirizzo di destinazione
- nel lookup P2P si parte dalla risorsa desiderata e si cerca il peer che può fornirla

Ad es. conosco identificativo e nome del file che cerco, ma non so chi ce l'ha --> faccio la richiesta, che verrà elaborata e inoltrata secondo le regole dell'overlay fino al peer adatto, per poi passare al retrieval (ciò ovviamente non sostituisce il routing IP, che resta necessario per trasportare i messaggi tra peer).

L'overlay deve inoltre supportare **ingresso e uscita dei peer**, **inserimento e rimozione risorse**, **identificazione nodi e risorse**.  

Per l'identificazione si ricorre a **GUID (Globally Unique Identifier)**, cioè identificatori univoci a livello globale per tutto il sistema.  
Una tecnica in questo senso è applicare una **funzione hash crittografica** a informazioni associate all'elemento (es. per la risorsa il contenuto, nome o altri metadati; per il peer informazioni identificative come IP e porta). Il risultato dell'hash è un **identificatore di lunghezza fissa** (quindi stavolta l'hash usata per id, non per cifrare!) che sarà univoco w.h.p.

Il GUID fornisce quindi un riferimento utilizzabile dal sistema, ma da solo non dice su quale peer si trova la risorsa: servono regole di organizzazione e lookup. 

##### Classificazione degli overlay
Come organizzare nodi e risorse e come instradare le richieste? La risposta dipende dal **tipo di overlay**:
- **Overlay non strutturato**: non impone topoogia globale prestabilita; i collegamenti derivano da decisioni locali e dal comportamento dei peer e la ricerca deve adattarsi a un'organizzazione relativamente libera
- **Overlay strutturato**: impone regole recise per organizzare nodi e risorse. I collegamenti seguono la struttura scelta e la ricerca può quindi sfruttare la topologia per essere più efficiente.

<img src="imgs/overlay_types.png" width="350"/>